# Red Object Segmentation — Full Colab Pipeline

**Runtime → Change runtime type → T4 GPU → Save**, then run cells top to bottom.

## Pipeline (keep each stage separate)

| Cell | Stage |
|------|-------|
| 1 | Install dependencies |
| 2 | Configuration |
| 3 | Search query definitions |
| 4 | Helper functions |
| 5 | **Download raw images** → `RAW_DIR/<search>/` |
| 6 | **Review raw downloads** — delete bad/irrelevant images |
| 7 | Preprocess — validate, dedupe, cap |
| 8 | HSV auto-annotation |
| 9 | Review annotations |
| 10+ | Dataset → train → YOLO auto-label → retrain → export |

**Important:** Use **specific** search queries (`"red apple fruit photo"`), not vague ones (`"red objects"`). Vague queries return random Bing junk. Set `REDOWNLOAD=True` in Cell 2 when you change queries, or old bad images are kept.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q ultralytics icrawler opencv-python-headless tqdm ipywidgets

import subprocess
subprocess.run(["jupyter", "nbextension", "enable", "--py", "widgetsnbextension", "--sys-prefix"], check=False)

print("Done.")

In [ ]:
# Cell 2 — Configuration
from pathlib import Path

import torch
if torch.cuda.is_available():
    print(f"GPU ready: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Training will be slow. Runtime > Change runtime type > T4 GPU")

# --- dataset size ---
TOTAL_IMAGES = 2000
IMAGES_PER_SEARCH = 80          # target per search (45 × 80 = 3600 theoretical max)
RANDOM_SEED = 42

# --- download ---
# Set to 2 while validating search queries. Set None to run all searches.
MAX_SEARCHES = 2
MAX_WORKERS = 8
REDOWNLOAD = True               # delete each search folder before downloading (needed after fixing queries)
WIPE_ALL_RAW = False            # set True once to delete entire RAW_DIR and start completely fresh

# Bing filters — force photo results that Bing classifies as red
BING_FILTERS = dict(type="photo", color="red", size="large")

# --- HSV red detection ---
MIN_RED_AREA = 20
POLYGON_EPSILON = 0.003
MIN_SATURATION = 60
MIN_VALUE = 40
SAVE_PREVIEWS = True

# --- training ---
MODEL = "yolov8n-seg.pt"
EPOCHS_ROUND1 = 50
EPOCHS_ROUND2 = 30
IMGSZ = 640
BATCH = 16
CONF_AUTO_LABEL = 0.35

# --- paths ---
ROOT = Path("/content/red_dataset")
RAW_DIR = ROOT / "raw"
UNIQUE_DIR = ROOT / "unique_images"
ANNOT_DIR = ROOT / "annotations"
PREVIEW_DIR = ROOT / "previews"
REVIEW_DIR = ROOT / "review"
YOLO_DIR = ROOT / "yolo"
AUTO_LABEL_DIR = ROOT / "auto_labels"
RUNS_DIR = Path("/content/runs")

for d in [RAW_DIR, UNIQUE_DIR, ANNOT_DIR, PREVIEW_DIR, REVIEW_DIR, YOLO_DIR, AUTO_LABEL_DIR, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {ROOT}")
print(f"MAX_SEARCHES = {MAX_SEARCHES!r}  (None = all searches)")
print(f"REDOWNLOAD = {REDOWNLOAD}  (delete folder before each search — turn off after queries validated)")

In [ ]:
# Cell 3 — Search query definitions
#
# Use CONCRETE nouns, not vague "red objects". Bing ignores vague queries and
# returns random trending images (often cats/memes). Each query should name
# a specific red thing + "photo" to bias toward real photographs.

SEARCHES = [
    # --- fruit / food (good red colour reference) ---
    "red apple fruit photo",
    "red tomato vegetable photo",
    "red strawberry fruit photo",
    "red cherry fruit photo",
    "red bell pepper photo",
    "red chili pepper photo",
    "red pomegranate fruit photo",

    # --- toys / household ---
    "red ball toy photo",
    "red lego brick photo",
    "red plastic cup photo",
    "red mug cup photo",
    "red bucket photo",
    "red balloon photo",
    "red toy car photo",
    "red dice photo",
    "red pencil photo",
    "red book cover photo",

    # --- tools / safety / outdoor ---
    "red fire extinguisher photo",
    "red stop sign photo",
    "red traffic cone photo",
    "red toolbox photo",
    "red first aid kit photo",
    "red mailbox photo",
    "red life buoy ring photo",
    "red fishing float bobber photo",
    "red gas can photo",
    "red hard hat photo",

    # --- clothing / bags ---
    "red hat cap photo",
    "red shoes sneakers photo",
    "red backpack bag photo",
    "red jacket photo",
    "red gloves photo",
    "red umbrella photo",

    # --- packaging / containers ---
    "red soda can photo",
    "red wine bottle photo",
    "red cardboard box photo",
    "red paint can photo",
    "red plastic bottle photo",

    # --- nature (still red objects) ---
    "red rose flower photo",
    "red maple leaf photo",
    "red mushroom fungus photo",

    # --- size / visibility variants ---
    "small red object close up photo",
    "tiny red item macro photo",
    "red object white background photo",
    "red object on table photo",
    "multiple red apples photo",
    "partially visible red object photo",
    "dark red object low light photo",
    "bright red object sunlight photo",

    # --- confusing colours (teach model what NOT to call red) ---
    "orange fruit photo isolated",
    "pink flower photo isolated",
    "purple grape photo isolated",
    "brown shoe photo isolated",
    "yellow banana photo isolated",
    "green apple photo isolated",
]

print(f"{len(SEARCHES)} search queries defined.")
print("First 3:", SEARCHES[:3])
if MAX_SEARCHES:
    print(f"Cell 5 will run first {MAX_SEARCHES} only (change MAX_SEARCHES in Cell 2 to expand).")

In [ ]:
# Cell 4 — Helper functions
import re
import cv2
import json
import hashlib
import random
import shutil
import numpy as np
from tqdm.notebook import tqdm

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def safe_name(text):
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", text).strip("_")


def count_images(folder):
    return sum(1 for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)


def list_raw_images(folder=None):
    root = folder or RAW_DIR
    return sorted(p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)


def sha256_file(path):
    h = hashlib.sha256()
    try:
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return None


def is_valid_image(path):
    try:
        img = cv2.imread(str(path))
        if img is None:
            return False
        h, w = img.shape[:2]
        return w >= 100 and h >= 100
    except Exception:
        return False


def create_red_mask(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lower_red_1 = np.array([0, MIN_SATURATION, MIN_VALUE])
    upper_red_1 = np.array([10, 255, 255])
    lower_red_2 = np.array([170, MIN_SATURATION, MIN_VALUE])
    upper_red_2 = np.array([179, 255, 255])
    mask1 = cv2.inRange(hsv, lower_red_1, upper_red_1)
    mask2 = cv2.inRange(hsv, lower_red_2, upper_red_2)
    mask = cv2.bitwise_or(mask1, mask2)
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    return mask


def mask_to_polygons(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    height, width = mask.shape
    polygons = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < MIN_RED_AREA:
            continue
        perimeter = cv2.arcLength(contour, True)
        epsilon = POLYGON_EPSILON * perimeter
        polygon = cv2.approxPolyDP(contour, epsilon, True)
        if len(polygon) < 3:
            continue
        points = polygon.reshape(-1, 2)
        normalized = []
        for x, y in points:
            normalized.extend([x / width, y / height])
        if len(normalized) >= 6:
            polygons.append(normalized)
    return polygons


def save_yolo_label(label_path, polygons, class_id=0):
    with open(label_path, "w") as f:
        for polygon in polygons:
            values = [str(class_id)] + [f"{v:.6f}" for v in polygon]
            f.write(" ".join(values) + "\n")


def save_preview(image, mask, polygons, output_path):
    preview = image.copy()
    overlay = np.zeros_like(image)
    overlay[:, :, 2] = mask
    preview = cv2.addWeighted(preview, 0.75, overlay, 0.25, 0)
    height, width = mask.shape
    for polygon in polygons:
        points = []
        for i in range(0, len(polygon), 2):
            points.append([int(polygon[i] * width), int(polygon[i + 1] * height)])
        pts = np.array(points, dtype=np.int32)
        cv2.polylines(preview, [pts], True, (0, 255, 0), 2)
    ys, xs = np.where(mask > 0)
    if len(xs) > 0:
        x1, x2 = int(xs.min()), int(xs.max())
        y1, y2 = int(ys.min()), int(ys.max())
        cv2.rectangle(preview, (x1, y1), (x2, y2), (255, 0, 255), 3)
        cv2.putText(preview, "ALL RED", (x1, max(30, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 255), 2)
    cv2.imwrite(str(output_path), preview)


def load_review_state(path, items):
    if path.exists():
        with open(path) as f:
            state = json.load(f)
    else:
        state = {"approved": [], "rejected": [], "index": 0}
    valid_names = {p.name for p, _ in items} if items and isinstance(items[0], tuple) else {p.name for p in items}
    state["approved"] = [n for n in state.get("approved", []) if n in valid_names]
    state["rejected"] = [n for n in state.get("rejected", []) if n in valid_names]
    state["index"] = min(state.get("index", 0), max(len(items) - 1, 0))
    return state


def save_review_state(path, state):
    with open(path, "w") as f:
        json.dump(state, f, indent=2)


print("Helper functions loaded.")

In [ ]:
# Cell 5 — Download raw images (BingImageCrawler only)
# Job: SEARCHES → Bing → RAW_DIR/<search>/
# Does NOT: annotate, train, filter by quality, or create splits.

from icrawler.builtin import BingImageCrawler

RAW_DIR.mkdir(parents=True, exist_ok=True)
TARGET_PER_SEARCH = IMAGES_PER_SEARCH

if WIPE_ALL_RAW:
    print("WIPE_ALL_RAW=True — deleting entire RAW_DIR")
    shutil.rmtree(RAW_DIR)
    RAW_DIR.mkdir(parents=True, exist_ok=True)

searches_to_run = SEARCHES[:MAX_SEARCHES] if MAX_SEARCHES else SEARCHES

print("=" * 60)
print("DOWNLOAD RAW IMAGES")
print("=" * 60)
print(f"Running {len(searches_to_run)}/{len(SEARCHES)} searches")
print(f"Target: {TARGET_PER_SEARCH} images per search")
print(f"Bing filters: {BING_FILTERS}")
print()

for number, search in enumerate(searches_to_run, start=1):
    folder_name = f"{number:02d}_{safe_name(search)}"
    out_dir = RAW_DIR / folder_name

    if REDOWNLOAD and out_dir.exists():
        print(f"[{number}/{len(searches_to_run)}] {search}")
        print(f"  REDOWNLOAD=True — clearing {out_dir.name}")
        shutil.rmtree(out_dir)

    out_dir.mkdir(parents=True, exist_ok=True)
    existing = count_images(out_dir)
    remaining = TARGET_PER_SEARCH - existing

    print(f"[{number}/{len(searches_to_run)}] {search}")
    print(f"  folder: {out_dir.name}")
    print(f"  existing: {existing}")

    if remaining <= 0:
        print(f"  already have {existing} images — skipping")
        print()
        continue

    print(f"  need {remaining} more images")
    print("  searching Bing...")

    try:
        before = count_images(out_dir)
        crawler = BingImageCrawler(
            storage={"root_dir": str(out_dir)},
            downloader_threads=MAX_WORKERS,
        )
        crawler.crawl(
            keyword=search,
            max_num=remaining,
            min_size=(200, 200),
            filters=BING_FILTERS,
        )
        after = count_images(out_dir)
        print(f"  Bing: +{after - before} new ({after} total in folder)")
    except Exception as e:
        print(f"  Bing error: {e}")

    print()

total_raw = len(list_raw_images())
print("=" * 60)
print(f"Download pass complete. Total raw images: {total_raw}")
print("Next: run Cell 6 — check results are red things, not cats/random junk.")
print("If still bad: set WIPE_ALL_RAW=True in Cell 2, re-run Cell 5.")

## Cell 6 — Review raw downloads

Bing returns **candidates**, not guaranteed good training data. Review what Cell 5 downloaded before running all 45 searches.

- Pick a search folder from the dropdown
- Browse pages of thumbnails in a grid
- **Delete** removes bad/irrelevant/corrupt files from `RAW_DIR`
- Re-run Cell 5 after improving `SEARCHES` if results are too noisy

In [ ]:
# Cell 6 — Raw image review (grid + delete)
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

COLS = 5


def search_folders():
    folders = sorted(d for d in RAW_DIR.iterdir() if d.is_dir())
    return folders or [RAW_DIR]


def images_in(folder):
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)


folder_dd = widgets.Dropdown(description="Folder")
page_slider = widgets.IntSlider(value=0, min=0, max=0, description="Page")
per_page = widgets.IntSlider(value=20, min=5, max=50, step=5, description="Per page")
btn_refresh = widgets.Button(description="Refresh grid")
delete_box = widgets.Text(placeholder="filename.jpg to delete", description="Delete")
btn_delete = widgets.Button(description="Delete file", button_style="danger")
btn_delete_page = widgets.Button(description="Delete ALL on page", button_style="warning")
status_raw = widgets.HTML()
out_raw = widgets.Output()


def refresh_folder_list():
    folders = search_folders()
    folder_dd.options = [(f.name, f) for f in folders]
    if folders:
        folder_dd.value = folders[0]


def current_page_images():
    folder = folder_dd.value
    imgs = images_in(folder)
    start = page_slider.value * per_page.value
    return imgs[start:start + per_page.value], len(imgs)


def update_page_slider():
    _, total = current_page_images() if folder_dd.value else ( [], 0)
    imgs = images_in(folder_dd.value) if folder_dd.value else []
    total = len(imgs)
    max_page = max(0, (total - 1) // per_page.value) if total else 0
    page_slider.max = max_page
    page_slider.value = min(page_slider.value, max_page)


def show_grid(_=None):
    clear_output(wait=True)
    with out_raw:
        folder = folder_dd.value
        imgs = images_in(folder)
        update_page_slider()
        page_imgs, _ = current_page_images()
        total = len(imgs)
        status_raw.value = (
            f"<b>{folder.name}</b> — {total} images &nbsp;|&nbsp; "
            f"page {page_slider.value + 1}/{page_slider.max + 1} &nbsp;|&nbsp; "
            f"showing {len(page_imgs)}"
        )
        if not page_imgs:
            print("No images in this folder.")
            return
        rows = (len(page_imgs) + COLS - 1) // COLS
        fig, axes = plt.subplots(rows, COLS, figsize=(3 * COLS, 3 * rows))
        axes = np.atleast_2d(axes)
        for i, ax in enumerate(axes.flat):
            ax.axis("off")
            if i >= len(page_imgs):
                continue
            path = page_imgs[i]
            img = cv2.imread(str(path))
            if img is not None:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.set_title(path.name[:18], fontsize=8)
        plt.suptitle(f"Raw review — {folder.name}", fontsize=12)
        plt.tight_layout()
        plt.show()
        print("Filenames on this page:")
        for p in page_imgs:
            print(f"  {p.name}")


def delete_file(path):
    if path.exists():
        path.unlink()
        return True
    return False


def on_delete(_=None):
    name = delete_box.value.strip()
    if not name:
        return
    target = folder_dd.value / name
    if delete_file(target):
        delete_box.value = ""
        show_grid()


def on_delete_page(_=None):
    page_imgs, _ = current_page_images()
    n = sum(delete_file(p) for p in page_imgs)
    print(f"Deleted {n} files from current page.")
    show_grid()


folder_dd.observe(lambda _: show_grid(), names="value")
page_slider.observe(lambda _: show_grid(), names="value")
per_page.observe(lambda _: (update_page_slider(), show_grid()), names="value")
btn_refresh.on_click(show_grid)
btn_delete.on_click(on_delete)
btn_delete_page.on_click(on_delete_page)

refresh_folder_list()
display(widgets.VBox([
    status_raw,
    widgets.HBox([folder_dd, per_page, page_slider, btn_refresh]),
    widgets.HBox([delete_box, btn_delete, btn_delete_page]),
    out_raw,
]))
show_grid()

In [ ]:
# Cell 7 — Preprocess: validate, dedupe, cap at TOTAL_IMAGES

print("=" * 60)
print("PREPROCESS — VALIDATE / DEDUPE / CAP")
print("=" * 60)

if UNIQUE_DIR.exists():
    shutil.rmtree(UNIQUE_DIR)
UNIQUE_DIR.mkdir(parents=True, exist_ok=True)

hashes = set()
unique_files = []
skipped_invalid = 0
skipped_dup = 0

for path in tqdm(list_raw_images(), desc="Scanning RAW_DIR"):
    if not is_valid_image(path):
        skipped_invalid += 1
        continue
    file_hash = sha256_file(path)
    if file_hash is None or file_hash in hashes:
        skipped_dup += 1
        continue
    hashes.add(file_hash)
    dest = UNIQUE_DIR / f"{len(unique_files):06d}{path.suffix.lower()}"
    shutil.copy2(path, dest)
    unique_files.append(dest)

print(f"Unique valid images: {len(unique_files)}")
print(f"Skipped invalid: {skipped_invalid}, duplicates: {skipped_dup}")

random.seed(RANDOM_SEED)
if len(unique_files) > TOTAL_IMAGES:
    unique_files = random.sample(unique_files, TOTAL_IMAGES)
    print(f"Capped to {TOTAL_IMAGES} images.")

print(f"Using {len(unique_files)} images for annotation.")

In [ ]:
# Cell 8 — HSV automatic pre-annotation

print("=" * 60)
print("HSV AUTOMATIC PRE-ANNOTATION")
print("=" * 60)

if ANNOT_DIR.exists():
    shutil.rmtree(ANNOT_DIR)
ANNOT_DIR.mkdir(parents=True, exist_ok=True)

if SAVE_PREVIEWS and PREVIEW_DIR.exists():
    shutil.rmtree(PREVIEW_DIR)
if SAVE_PREVIEWS:
    PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

annotated = []
empty_labels = 0

for image_path in tqdm(unique_files, desc="Annotating"):
    image = cv2.imread(str(image_path))
    if image is None:
        continue
    mask = create_red_mask(image)
    polygons = mask_to_polygons(mask)
    label_path = ANNOT_DIR / f"{image_path.stem}.txt"
    save_yolo_label(label_path, polygons)
    if not polygons:
        empty_labels += 1
    if SAVE_PREVIEWS:
        save_preview(image, mask, polygons, PREVIEW_DIR / image_path.name)
    annotated.append((image_path, label_path))

print(f"Annotated {len(annotated)} images ({empty_labels} with no red detected).")
print(f"Previews: {PREVIEW_DIR}")

## Cell 9 — Review HSV annotations

- **Approve** — image + label good (or intentionally empty = no red)
- **Reject** — excluded from training
- Only approved images used for round-1 training

In [ ]:
# Cell 9 — Interactive review: HSV pre-annotations
from IPython.display import display, clear_output
import ipywidgets as widgets

HSV_REVIEW_FILE = REVIEW_DIR / "hsv_review.json"
review_items = list(annotated)
hsv_state = load_review_state(HSV_REVIEW_FILE, review_items)

out = widgets.Output()
status = widgets.HTML()
btn_approve = widgets.Button(description="Approve", button_style="success", layout=widgets.Layout(width="120px"))
btn_reject = widgets.Button(description="Reject", button_style="danger", layout=widgets.Layout(width="120px"))
btn_skip = widgets.Button(description="Skip", layout=widgets.Layout(width="120px"))
btn_prev = widgets.Button(description="Prev", layout=widgets.Layout(width="80px"))
btn_next = widgets.Button(description="Next", layout=widgets.Layout(width="80px"))
jump_box = widgets.BoundedIntText(value=1, min=1, max=max(len(review_items), 1), description="Go to #")
btn_jump = widgets.Button(description="Jump", layout=widgets.Layout(width="80px"))


def show_current():
    clear_output(wait=True)
    with out:
        if not review_items:
            print("No images to review.")
            return
        idx = hsv_state["index"]
        img_path, lbl_path = review_items[idx]
        preview_path = PREVIEW_DIR / img_path.name
        img = cv2.imread(str(preview_path if preview_path.exists() else img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        n_poly = sum(1 for line in open(lbl_path) if line.strip()) if lbl_path.exists() else 0
        approved = len(hsv_state["approved"])
        rejected = len(hsv_state["rejected"])
        remaining = len(review_items) - approved - rejected
        status.value = (
            f"<b>{idx + 1}/{len(review_items)}</b> &nbsp;|&nbsp; "
            f"Approved: {approved} &nbsp; Rejected: {rejected} &nbsp; Remaining: {remaining}<br>"
            f"<code>{img_path.name}</code> &nbsp; polygons: {n_poly}"
        )
        plt.figure(figsize=(10, 7))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"HSV review — {img_path.name}")
        plt.show()


def set_decision(decision):
    if not review_items:
        return
    idx = hsv_state["index"]
    name = review_items[idx][0].name
    hsv_state["approved"] = [n for n in hsv_state["approved"] if n != name]
    hsv_state["rejected"] = [n for n in hsv_state["rejected"] if n != name]
    if decision == "approve":
        hsv_state["approved"].append(name)
    elif decision == "reject":
        hsv_state["rejected"].append(name)
    if idx < len(review_items) - 1:
        hsv_state["index"] += 1
    save_review_state(HSV_REVIEW_FILE, hsv_state)
    show_current()


def move(delta):
    if not review_items:
        return
    hsv_state["index"] = max(0, min(len(review_items) - 1, hsv_state["index"] + delta))
    save_review_state(HSV_REVIEW_FILE, hsv_state)
    show_current()


def jump_to(_=None):
    hsv_state["index"] = jump_box.value - 1
    save_review_state(HSV_REVIEW_FILE, hsv_state)
    show_current()


btn_approve.on_click(lambda _: set_decision("approve"))
btn_reject.on_click(lambda _: set_decision("reject"))
btn_skip.on_click(lambda _: move(1))
btn_prev.on_click(lambda _: move(-1))
btn_next.on_click(lambda _: move(1))
btn_jump.on_click(jump_to)

display(widgets.VBox([
    status,
    widgets.HBox([btn_prev, btn_approve, btn_reject, btn_skip, btn_next, jump_box, btn_jump]),
    out,
]))
show_current()

In [ ]:
# Cell 10 — Build round-1 YOLO dataset from approved HSV reviews

print("=" * 60)
print("BUILD ROUND-1 DATASET")
print("=" * 60)

with open(HSV_REVIEW_FILE) as f:
    hsv_state = json.load(f)

approved_names = set(hsv_state.get("approved", []))
if not approved_names:
    raise RuntimeError("No approved images. Run Cell 9 and approve some images first.")

approved_pairs = [(p, lbl) for p, lbl in annotated if p.name in approved_names]
print(f"Approved for training: {len(approved_pairs)}")

random.seed(RANDOM_SEED)
random.shuffle(approved_pairs)
n = len(approved_pairs)
train_end = int(n * 0.80)
val_end = int(n * 0.90)
splits = {
    "train": approved_pairs[:train_end],
    "val": approved_pairs[train_end:val_end],
    "test": approved_pairs[val_end:],
}

for split_name, pairs in splits.items():
    img_dir = YOLO_DIR / "images" / split_name
    lbl_dir = YOLO_DIR / "labels" / split_name
    if img_dir.exists():
        shutil.rmtree(img_dir)
    if lbl_dir.exists():
        shutil.rmtree(lbl_dir)
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for img_path, lbl_path in pairs:
        shutil.copy2(img_path, img_dir / img_path.name)
        shutil.copy2(lbl_path, lbl_dir / lbl_path.name)
    print(f"  {split_name}: {len(pairs)} images")

DATA_YAML = YOLO_DIR / "data.yaml"
DATA_YAML.write_text(f"""path: {YOLO_DIR.resolve()}
train: images/train
val: images/val
test: images/test
names:
  0: red
""")
print(f"\nCreated {DATA_YAML}")

In [ ]:
# Cell 11 — Train YOLO segmentation (round 1)
from ultralytics import YOLO

print("=" * 60)
print("TRAINING ROUND 1")
print("=" * 60)

model = YOLO(MODEL)
results1 = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS_ROUND1,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(RUNS_DIR),
    name="red_seg_round1",
    exist_ok=True,
    verbose=True,
)
BEST_ROUND1 = Path(results1.save_dir) / "weights" / "best.pt"
print(f"\nRound 1 complete: {BEST_ROUND1}")

In [ ]:
# Cell 12 — YOLO auto-labels remaining images

print("=" * 60)
print("YOLO AUTO-LABEL ADDITIONAL IMAGES")
print("=" * 60)

with open(HSV_REVIEW_FILE) as f:
    hsv_state = json.load(f)

reviewed = set(hsv_state.get("approved", [])) | set(hsv_state.get("rejected", []))
unreviewed = [p for p, _ in annotated if p.name not in reviewed]
pool = unreviewed if len(unreviewed) >= 10 else [p for p, _ in annotated if p.name not in hsv_state.get("approved", [])]
print(f"Images to auto-label: {len(pool)}")

if AUTO_LABEL_DIR.exists():
    shutil.rmtree(AUTO_LABEL_DIR)
AUTO_IMAGES = AUTO_LABEL_DIR / "images"
AUTO_LABELS = AUTO_LABEL_DIR / "labels"
AUTO_PREVIEWS = AUTO_LABEL_DIR / "previews"
for d in [AUTO_IMAGES, AUTO_LABELS, AUTO_PREVIEWS]:
    d.mkdir(parents=True, exist_ok=True)

auto_model = YOLO(str(BEST_ROUND1))
auto_pairs = []

for img_path in tqdm(pool, desc="Auto-labeling"):
    dest_img = AUTO_IMAGES / img_path.name
    shutil.copy2(img_path, dest_img)
    results = auto_model.predict(source=str(dest_img), conf=CONF_AUTO_LABEL, imgsz=IMGSZ, verbose=False)
    label_path = AUTO_LABELS / f"{img_path.stem}.txt"
    image = cv2.imread(str(dest_img))
    h, w = image.shape[:2]
    polygons = []
    if results and results[0].masks is not None:
        for mask_xy in results[0].masks.xyn:
            flat = mask_xy.reshape(-1).tolist()
            if len(flat) >= 6:
                polygons.append(flat)
    save_yolo_label(label_path, polygons)
    mask = np.zeros((h, w), dtype=np.uint8)
    if results and results[0].masks is not None:
        for seg in results[0].masks.xy:
            cv2.fillPoly(mask, [seg.astype(np.int32)], 255)
    save_preview(image, mask, polygons, AUTO_PREVIEWS / img_path.name)
    auto_pairs.append((dest_img, label_path))

with_labels = sum(1 for _, lbl in auto_pairs if lbl.stat().st_size > 0)
print(f"Auto-labeled {len(auto_pairs)} images ({with_labels} with detections).")

## Cell 13 — Review YOLO auto-labels

Only **approved** auto-labels merge into the round-2 dataset.

In [ ]:
# Cell 13 — Interactive review: YOLO auto-labels
YOLO_REVIEW_FILE = REVIEW_DIR / "yolo_review.json"
yolo_review_items = list(auto_pairs)
yolo_state = load_review_state(YOLO_REVIEW_FILE, yolo_review_items)

out2 = widgets.Output()
status2 = widgets.HTML()
y_approve = widgets.Button(description="Approve", button_style="success", layout=widgets.Layout(width="120px"))
y_reject = widgets.Button(description="Reject", button_style="danger", layout=widgets.Layout(width="120px"))
y_skip = widgets.Button(description="Skip", layout=widgets.Layout(width="120px"))
y_prev = widgets.Button(description="Prev", layout=widgets.Layout(width="80px"))
y_next = widgets.Button(description="Next", layout=widgets.Layout(width="80px"))


def show_yolo_current():
    clear_output(wait=True)
    with out2:
        if not yolo_review_items:
            print("No auto-labeled images to review.")
            return
        idx = yolo_state["index"]
        img_path, lbl_path = yolo_review_items[idx]
        preview_path = AUTO_PREVIEWS / img_path.name
        img = cv2.imread(str(preview_path if preview_path.exists() else img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        n_poly = sum(1 for line in open(lbl_path) if line.strip()) if lbl_path.exists() else 0
        status2.value = (
            f"<b>{idx + 1}/{len(yolo_review_items)}</b> &nbsp;|&nbsp; "
            f"Approved: {len(yolo_state['approved'])} &nbsp; Rejected: {len(yolo_state['rejected'])}<br>"
            f"<code>{img_path.name}</code> &nbsp; polygons: {n_poly}"
        )
        plt.figure(figsize=(10, 7))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"YOLO auto-label review — {img_path.name}")
        plt.show()


def yolo_decision(decision):
    idx = yolo_state["index"]
    name = yolo_review_items[idx][0].name
    yolo_state["approved"] = [n for n in yolo_state["approved"] if n != name]
    yolo_state["rejected"] = [n for n in yolo_state["rejected"] if n != name]
    if decision == "approve":
        yolo_state["approved"].append(name)
    elif decision == "reject":
        yolo_state["rejected"].append(name)
    if idx < len(yolo_review_items) - 1:
        yolo_state["index"] += 1
    save_review_state(YOLO_REVIEW_FILE, yolo_state)
    show_yolo_current()


def yolo_move(delta):
    yolo_state["index"] = max(0, min(len(yolo_review_items) - 1, yolo_state["index"] + delta))
    save_review_state(YOLO_REVIEW_FILE, yolo_state)
    show_yolo_current()


y_approve.on_click(lambda _: yolo_decision("approve"))
y_reject.on_click(lambda _: yolo_decision("reject"))
y_skip.on_click(lambda _: yolo_move(1))
y_prev.on_click(lambda _: yolo_move(-1))
y_next.on_click(lambda _: yolo_move(1))

display(widgets.VBox([status2, widgets.HBox([y_prev, y_approve, y_reject, y_skip, y_next]), out2]))
show_yolo_current()

In [ ]:
# Cell 14 — Build round-2 dataset + retrain

with open(HSV_REVIEW_FILE) as f:
    hsv_state = json.load(f)
with open(YOLO_REVIEW_FILE) as f:
    yolo_state = json.load(f)

hsv_approved = {p.name: (p, lbl) for p, lbl in annotated if p.name in hsv_state.get("approved", [])}
yolo_approved = {p.name: (p, lbl) for p, lbl in auto_pairs if p.name in yolo_state.get("approved", [])}
merged = {**hsv_approved, **yolo_approved}
merged_pairs = list(merged.values())
print(f"Round-2 dataset: {len(merged_pairs)} images")

random.seed(RANDOM_SEED)
random.shuffle(merged_pairs)
n = len(merged_pairs)
train_end, val_end = int(n * 0.80), int(n * 0.90)
splits2 = {"train": merged_pairs[:train_end], "val": merged_pairs[train_end:val_end], "test": merged_pairs[val_end:]}

YOLO_DIR2 = ROOT / "yolo_round2"
for split_name, pairs in splits2.items():
    img_dir = YOLO_DIR2 / "images" / split_name
    lbl_dir = YOLO_DIR2 / "labels" / split_name
    if img_dir.exists():
        shutil.rmtree(img_dir)
    if lbl_dir.exists():
        shutil.rmtree(lbl_dir)
    img_dir.mkdir(parents=True)
    lbl_dir.mkdir(parents=True)
    for img_path, lbl_path in pairs:
        shutil.copy2(img_path, img_dir / img_path.name)
        shutil.copy2(lbl_path, lbl_dir / lbl_path.name)
    print(f"  {split_name}: {len(pairs)}")

DATA_YAML2 = YOLO_DIR2 / "data.yaml"
DATA_YAML2.write_text(f"""path: {YOLO_DIR2.resolve()}
train: images/train
val: images/val
test: images/test
names:
  0: red
""")

print("\nRetraining round 2...")
model2 = YOLO(str(BEST_ROUND1))
results2 = model2.train(
    data=str(DATA_YAML2), epochs=EPOCHS_ROUND2, imgsz=IMGSZ, batch=BATCH,
    project=str(RUNS_DIR), name="red_seg_round2", exist_ok=True, verbose=True,
)
BEST_FINAL = Path(results2.save_dir) / "weights" / "best.pt"
print(f"Final model: {BEST_FINAL}")

In [ ]:
# Cell 15 — Validate + export for Raspberry Pi
from google.colab import files

final_model = YOLO(str(BEST_FINAL))
metrics = final_model.val(data=str(DATA_YAML2), imgsz=IMGSZ, verbose=False)
print("\n=== Final Validation ===")
try:
    print(f"  mAP50     : {metrics.seg.map50:.4f}")
    print(f"  mAP50-95  : {metrics.seg.map:.4f}")
    print(f"  Precision : {metrics.seg.mp:.4f}")
    print(f"  Recall    : {metrics.seg.mr:.4f}")
except AttributeError:
    pass

PI_EXPORT = Path("/content/red_seg_pi_export")
if PI_EXPORT.exists():
    shutil.rmtree(PI_EXPORT)
PI_EXPORT.mkdir()
shutil.copy2(BEST_FINAL, PI_EXPORT / "best.pt")
(PI_EXPORT / "README.txt").write_text(
    "Copy best.pt to ~/yolo-project/weights/best.pt\n"
    "Set architecture: yolov8n-seg in config/model.yaml\n"
)
shutil.make_archive("/content/red_seg_pi_export", "zip", PI_EXPORT)
files.download(str(BEST_FINAL))
files.download("/content/red_seg_pi_export.zip")
print("Downloaded best.pt + export zip.")